# 55 — Form-defined families (Task 6, exploratory)

Tests for the convergence of a dominant design on families declared by **form** (wing, tilt, booms, propulsor band, enclosure), not by architecture.
Exploratory, after Task 1, on the **w15** weighting and nothing else. Nothing here changes F or any number of the dissertation: F stays as decided on 2026-10-14.
The brief named this notebook 25; 53 and 54 already exist, so it is 55.

All the logic is in `viz/thesis_form_families.py` (pure functions over `ctx`, each cached on the context); this notebook only calls it, shows each table and writes the outputs.

| step | function | output |
|---|---|---|
| 6.1 form variables | `form_variables` | `q8_form_variables.csv` |
| 6.2 declared families | `FAMILIES`, `membership` | (rules fixed before running) |
| 6.3a membership | `membership_tables`, `crosstab` | `q8_families_membership.csv` |
| 6.3b premise | `premise_check`, `q_decomposition` | `q8_families_premise.csv`, `q8_families_q_decomposition.csv` |
| 6.3c condition 1 | `condition1` | `q8_families_condition1.csv` |
| 6.3d condition 2 | `condition2` | `q8_families_condition2.csv` |
| 6.3e condition 3 | `condition3` (quoted from Task 1) | `q8_families_condition3.csv` |
| 6.4 verdict | `verdict_table`, `numbers_md` | `q8_families_verdict.csv`, `NUMBERS_VI_form_families.md` |

Outputs: `figures/Part_III_dominant_design/III3_form_families/exploratory/`, copies in `latex/Data_for_writing/csv/` (md in `latex/Data_for_writing/`) and in `handoff_2026-10-16/task6/` at the repository root.


## Setup

In [ ]:
import sys, pathlib, importlib
PILLAR = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(PILLAR / "src"))
import pandas as pd
from IPython.display import display, Markdown
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60); pd.set_option('display.max_colwidth', 120)
from viz import config, thesis
ff = importlib.import_module('viz.thesis_form_families')
cfg = config.load()
ctx = thesis.context(cfg)
OUT = ff.out_dir(cfg)
print(len(ctx.v), 'designs | cohorts', ctx.WIN_NAMES, '| seed', ctx.SEED, '| rarefy', cfg.analysis['rarefy_n'], '| permutations', ff.PERMS, '| writing to', OUT)

## 6.1 Form variables

One row per design with an architecture (625). `boom_n` is the raw boom total (not the w15 rank); the band is the w15 propulsor band as a label; the rest come from the w15 frame. A blank means an override hides the reading or the design has no propulsor card / count.

In [ ]:
fv = ff.form_variables(ctx)
print(fv.shape, '| blanks:', {c: int(n) for c, n in fv.isna().sum().items() if n})
display(fv.head(10))
for c in ['wing_n', 'wing_tilts', 'band', 'enclosure', 'empType']:
    print(c, fv[c].value_counts(dropna=False).to_dict())

## 6.2 Declared families

Fixed before running; none added or tuned after seeing results. Empennage type is never a membership rule. A design whose rule variable is blank is not a member.

In [ ]:
display(ff.declared_families())
mem = ff.membership(ctx)
print('members:', mem[ff.FAMILY_IDS].sum().to_dict())
print('not members for a blank rule variable:', {f: int(mem[f + '_blank'].sum()) for f in ff.FAMILY_IDS})

## 6.3a Membership

Family x architecture (codes), designs per cohort and the empennage distribution inside each family, on both bases. A family more than 85 % one architecture is that architecture renamed.

In [ ]:
mt = ff.membership_tables(ctx)
display(mt['summary'].drop(columns=['architectures (code: designs)']))
print('all designs'); display(ff.crosstab(ctx, 'all designs'))
print('actively protected'); display(ff.crosstab(ctx, 'actively protected'))
long = mt['long']
display(long[long['kind'].eq('cohort') & long['base'].eq('all designs')].pivot(index='family', columns='value', values='designs')[ctx.WIN_NAMES])
display(long[long['kind'].eq('empennage') & long['base'].eq('all designs')].pivot(index='family', columns='value', values='designs').fillna(0).astype(int))

## 6.3b Premise check

Per cohort and base, 200 random subsets of the cohort's designs of the family's size (seed 42): cohesion = mean w15 Gower distance within the family against the band of the subsets; separation = mean distance family-to-rest against the same band. The premise holds when cohesion is below the band AND separation is above it. Then Q of each complete cohort (all designs) split into within-family, within-rest and between pairs, to see which pairs carry the ΔQ of Task 1.

In [ ]:
pr = ff.premise_check(ctx)
display(pr[pr['cohort'].eq('2020-23')].drop(columns=['partial']))
display(pr[pr['read'].astype(bool)][['family', 'base', 'cohort', 'family designs', 'cohesion', 'cohesion band low', 'separation', 'separation band high', 'premise holds']])
qd = ff.q_decomposition(ctx)
display(qd[qd['cohort'].eq('2020-23')])

## 6.3c Condition 1

The family's share per priority year, read years only (20 designs or more), met above 50 % in two consecutive years; the same code path as `q7_condition1_family`. Three consecutive years as sensitivity.

In [ ]:
c1 = ff.condition1(ctx)
y = c1['yearly']
display(y[y['read'].astype(bool)].pivot(index='priority year', columns=['base', 'family'], values='family share'))
for k, m in c1['met'].items():
    print(k, '| two consecutive years:', m['two'] or 'none', '| three:', m['three'] or 'none')

## 6.3d Condition 2

²D rarefied to 40 against the 200-permutation band (seed 42, fresh generator per family and base) on the partition {family} ∪ {each remaining architecture as its own option}, exactly as F was computed.

In [ ]:
c2 = ff.condition2(ctx)
display(c2[~c2['partial']][['family', 'base', 'partition', 'cohort', 'designs', 'top option', 'top share', 'D2', 'D2 permutation low', 'D2 permutation high', 'below band']])

## 6.3e Condition 3

Quoted from Task 1 per base (w15, complete cohorts): Rao's Q is cohort-wide and blind to the family, so it is never recomputed per family. This step loads the Task 1 cache (a few minutes).

In [ ]:
c3 = ff.condition3(ctx)
display(c3)

## 6.4 Verdict

The shape of `q7_verdict_table.csv` with the cohesion and separation rows; "Met" only when all three hold in the same period on the same base.

In [ ]:
vt = ff.verdict_table(ctx)
display(vt)
for base, _ in ff.BASES:
    print(base); display(ff.verdict_md_table(ctx, base))

## Write the outputs

Results folder, `latex/Data_for_writing/csv/` (+ the md in `Data_for_writing/`) and `handoff_2026-10-16/task6/` (every output plus `viz/thesis_form_families.py` and this notebook).

In [ ]:
written = ff.write_outputs(ctx, 'task6')
print('\n'.join(map(str, written)))
display(Markdown((OUT / 'NUMBERS_VI_form_families.md').read_text(encoding='utf-8')))